In [9]:
import torch
import random
import math
from torch import nn

In [2]:
def train_model(model, train_input, train_target, mini_batch_size):
    
    criterion = nn.MSELoss()
    eta = 1e-1

    for e in range(25):
        sum_loss = 0
        for b in range(0, train_input.size(0), mini_batch_size):
            output = model(train_input.narrow(0, b, mini_batch_size))
            loss = criterion(output, train_target.narrow(0, b, mini_batch_size))
            model.zero_grad()
            loss.backward()
            sum_loss = sum_loss + loss.item()
            with torch.no_grad():
                for p in model.parameters():
                    p -= eta * p.grad
                    
        print(e, sum_loss)

In [10]:
# generate 2d points in [0,1] squared, targets are 0 if point inside the circle of squared radius 1/2pi and 1 outside.
# return coordinates and target tensors, both of size Nx2, plus classes tensor of size Nx1
def generate_points_classes(nb):
    input = torch.empty((nb, 2))
    target = torch.empty((nb, 2))
    classes = torch.empty((nb, 1))
    for i in range(nb):
        x = random.uniform(0,1) - 0.5
        y = random.uniform(0,1) - 0.5
        t = int(2 * math.pi * (pow(x, 2) + pow(y, 2)) < 1)
        input[i] = torch.tensor([x + 0.5, y + 0.5])
        target[i,t] = 1
        target[i,1-t] = 0
        classes[i] = t
    return input, target, classes

In [47]:
# accuracy for classes prediction
def accuracy_classes(model, input, classes):
    
    nb_samples = input.shape[0]
    pred = model(input)
    _, pred_classes = pred.max(1)
    
    nb_errors = (pred_classes - classes[:,0]).type(torch.BoolTensor).sum().item()
    return (nb_samples - nb_errors) / nb_samples

In [69]:
model = nn.Sequential(
    nn.Linear(2, 25), 
    nn.ReLU(), 
    nn.Linear(25, 25),
    nn.ReLU(),
    nn.Linear(25, 25),
    nn.ReLU(), 
    nn.Linear(25, 2))

In [70]:
input, target, classes = generate_points_classes(1000)
input[:2, :], target[:2, :]

(tensor([[0.3864, 0.0610],
         [0.3366, 0.7437]]),
 tensor([[1., 0.],
         [0., 1.]]))

In [71]:
train_model(model, input, target, 10)

0 26.372991234064102
1 24.559789791703224
2 24.179878309369087
3 23.415275260806084
4 21.761940583586693
5 18.779141932725906
6 15.00277442485094
7 12.042807461693883
8 10.052849046885967
9 8.994267838075757
10 8.257999867200851
11 7.725395118817687
12 7.388750984333456
13 7.101890059188008
14 6.879165196791291
15 6.695213523693383
16 6.553683935198933
17 6.3849509190768
18 6.211342805996537
19 6.114339358638972
20 5.980692236684263
21 5.911547617521137
22 5.831859912723303
23 5.74685294367373
24 5.639124958310276


In [72]:
accuracy_classes(model, input, classes)

0.883